In [ ]:
# import depsi
import xarray as xr
from pathlib import Path
from depsi.io import read_slc_stack
from depsi.classification import ps_selection, network_stm_selection
import json

import fsspec
from dcachefs import dCacheFileSystem

## setup dCache connection

following this doc: https://dcachefs.readthedocs.io/en/latest/notebooks/user_manual.html

In [ ]:
# setup dCache connection
# following this doc: https://dcachefs.readthedocs.io/en/latest/notebooks/user_manual.html
path_macaroon_config = Path('/home/caroline-oku/macaroon/maca_caroline.conf')
with open(path_macaroon_config, "r", encoding="utf-8") as f:
    data = {}
    for line in f:
        line = line.strip()
        if not line or line.startswith(("#", ";")):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        data[key.strip()] = value.strip()
token = data['bearer_token']

In [ ]:
# api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1'
# fs = dCacheFileSystem(api_url=api_url, token=token)
# fs.ls('/disk/Data/')

In [ ]:
# Small data ~70MB
# path_stack = 'dcache://disk/Data/ou_dcache_benchmark/nl_amsterdam_s1_asc_t161.zarr'

In [ ]:
# Large data ~4.4GB
path_stack = 'dcache://disk/Data/ou_dcache_benchmark/nl_assendelft_s1_dsc_t037.zarr'

# Using a Dask SLURM cluster with 6 workers = 24 cores + 192GB mem
from dask.distributed import Client

client = Client("tcp://10.0.2.69:43581")
client

In [ ]:
zarr_store = fsspec.get_mapper(
    path_stack,
    token=token,
    api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1',
    webdav_url='https://webdav.grid.surfsara.nl:2880/',
)

In [ ]:
%%time
# Lazy loading stack
stack = read_slc_stack(zarr_store) # here depsi io is manually modfified to be able to work with zarr

# PS selection
stm_preselention_pnts = ps_selection(
    stack,
    method="nmad",
    threshold=0.5,  # Points with NMAD below this threshold are selected
    output_chunks=2500,
    mem_persist=False,
)
print(f"{stm_preselention_pnts.sizes["space"]} preselction points selected")


# Network pnts selection
mask = stm_preselention_pnts["full_ts_nmad"] < 0.2
stm_network_pnts = stm_preselention_pnts.where(mask.compute(), drop=True)
stm_network_pnts = network_stm_selection(
    stm_network_pnts,
    min_dist=100,
    azimuth_spacing=10,
    range_spacing=5,
    sortby_var="full_ts_nmad",  # The sort is in ascending order so that points with smaller NMAD are prioritized
)
print(f"{stm_network_pnts.sizes["space"]} preselction points selected")

# Write preselection and network points to zarr
preselection_out = fsspec.get_mapper(
    "dcache://disk/Data/ou_dcache_benchmark/preselection.zarr",
    token=token,
    api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1',
    webdav_url='https://webdav.grid.surfsara.nl:2880/',
)
network_out = fsspec.get_mapper(
    "dcache://disk/Data/ou_dcache_benchmark/stm_network_pnts.zarr",
    token=token,
    api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1',
    webdav_url='https://webdav.grid.surfsara.nl:2880/',
)

stm_preselention_pnts.to_zarr(preselection_out, mode="w")
stm_network_pnts.to_zarr(network_out, mode="w")

small dataset: 

```
27458 preselction points selected
77preselction points selected
CPU times: user 11.4 s, sys: 956 ms, total: 12.3 s
Wall time: 1min 32s
```

large data

```
1785231 preselction points selected
92 preselction points selected
CPU times: user 5.37 s, sys: 1.21 s, total: 6.58 s
Wall time: 2min 39s
```